In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os
import json
import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")
    
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
CACHE_DIR = BASE_DIR / "LSTM_cache_TL/Transformer_exp"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
}
# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

# Transform data to monthly format (run or load data):
paths_HMA = {
    "era5_climate_data":
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
}
# Check that all these files exists
for key, path in paths_HMA.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:

In [ ]:
experiment_region_groups = {
    # "CEU": ["FR", "IT_AT", "CH"],
    "HMA": ["CENTRALASIA", "SOUTHASIAWEST", "SOUTHASIAEAST"]
}
paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

# Only the regions we care about for the pooled model
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', 'FR']

# Step 1: only build monthly cache for these three
run_flag_by_code = {
    'CH': False,
    'ISL': False,
    'NOR': False,
    'FR': False,
}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=SOURCE_REGIONS,  # <-- only three codes, no SJM/ALA/CENTRALASIA
    run_flag_by_code=run_flag_by_code,
)

# Step 2: no XREG_PAIRS needed — just get the train/test split per region
# res_xreg_by_source[region] gives you df_train, df_test, df_train_aug, df_test_aug
res_xreg_by_source = {
    region: monthly_cache[region]  # already has the train/test split
    for region in SOURCE_REGIONS
}

## Datasets:

### Ft and pool glaciers:

In [ ]:
save_path = BASE_DIR / "glacier_splits.json"

with open(save_path, "r") as f:
    output = json.load(f)

splits = {
    src_region: {
        "pool_glaciers": data["holdout_glaciers"],  # swapped
        "holdout_glaciers": data["pool_glaciers"],  # swapped
        "actual_pool_frac":
        1.0 - data["actual_pool_frac"],  # update fraction too
        "sinkhorn_pool_vs_holdout": data["sinkhorn_pool_vs_holdout"],
        "sinkhorn_pool_vs_region":
        data["sinkhorn_holdout_vs_region"],  # swapped
        "sinkhorn_holdout_vs_region":
        data["sinkhorn_pool_vs_region"],  # swapped
        "blur_joint": data["blur_joint"],
        "best_seed": data["best_seed"],
        "best_score": data["best_score"],
    }
    for src_region, data in output.items()
}

print(f"Loaded splits from: {save_path}")
for src_region, split in splits.items():
    print(f"\n{src_region}:")
    print(f"  Pool glaciers     : {split['pool_glaciers']}")
    print(f"  Holdout glaciers  : {split['holdout_glaciers']}")
    print(f"  Pool fraction     : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout)  : {split['sinkhorn_pool_vs_holdout']:.4f}")

### Sequence datasets:

In [ ]:
def build_pooled_assets_from_cache(
    res_xreg_by_source: dict,
    splits: dict,
    source_regions: list,
    monthly_cols: list,
    static_cols: list,
    cfg,
    seed: int = 42,
    val_ratio: float = 0.2,
    cache_dir: str = "logs/LSTM_cache_pooled",
    force_recompute: bool = False,
):
    import joblib
    os.makedirs(cache_dir, exist_ok=True)

    # cache key encodes which regions are pooled
    regions_key = "_".join(sorted(source_regions))
    train_p = os.path.join(cache_dir, f"pooled_{regions_key}_train.joblib")
    split_p = os.path.join(cache_dir, f"pooled_{regions_key}_split.joblib")

    # one cache file per holdout region
    holdout_ps = {
        region: os.path.join(cache_dir, f"holdout_{region}.joblib")
        for region in source_regions
    }

    all_cached = (os.path.exists(train_p) and os.path.exists(split_p)
                  and all(os.path.exists(p) for p in holdout_ps.values()))

    if all_cached and not force_recompute:
        print(f"Loading pooled assets from cache: {cache_dir}")
        ds_pooled = joblib.load(train_p)
        split = joblib.load(split_p)
        holdout_datasets = {
            region: joblib.load(p)
            for region, p in holdout_ps.items()
        }
        print(
            f"Pooled dataset: {len(ds_pooled)} sequences "
            f"({len(split['train_idx'])} train | {len(split['val_idx'])} val)")
        for region, ds_h in holdout_datasets.items():
            print(f"  {region} holdout: {len(ds_h)} sequences")
        return (
            {
                "ds_train": ds_pooled,
                "train_idx": split["train_idx"],
                "val_idx": split["val_idx"]
            },
            holdout_datasets,
        )

    # --- build fresh ---
    print(f"Building pooled assets for regions: {source_regions}")
    train_dfs_loss, train_dfs_full = [], []
    holdout_datasets = {}
    months_head_pad = months_tail_pad = None

    for region in source_regions:
        df_loss = res_xreg_by_source[region]['data_monthly']
        df_full = res_xreg_by_source[region]['data_monthly_aug']
        months_head_pad = res_xreg_by_source[region]['months_head_pad']
        months_tail_pad = res_xreg_by_source[region]['months_tail_pad']

        holdout_glaciers = splits[region]['holdout_glaciers']

        df_loss_train = df_loss[~df_loss['GLACIER'].isin(holdout_glaciers)]
        df_full_train = df_full[~df_full['GLACIER'].isin(holdout_glaciers)]
        df_loss_hold = df_loss[df_loss['GLACIER'].isin(holdout_glaciers)]
        df_full_hold = df_full[df_full['GLACIER'].isin(holdout_glaciers)]

        n_total = df_loss['GLACIER'].nunique()
        n_train = df_loss_train['GLACIER'].nunique()
        n_hold = df_loss_hold['GLACIER'].nunique()
        print(
            f"{region}: {n_total} glaciers → {n_train} train | {n_hold} holdout"
        )

        train_dfs_loss.append(df_loss_train)
        train_dfs_full.append(df_full_train)

        ds_holdout = build_combined_LSTM_dataset(
            df_loss=df_loss_hold,
            df_full=df_full_hold,
            monthly_cols=monthly_cols,
            static_cols=static_cols,
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            normalize_target=True,
            expect_target=True,
            show_progress=False,
        )
        holdout_datasets[region] = ds_holdout
        joblib.dump(ds_holdout, holdout_ps[region], compress=3)
        print(f"  Saved {region} holdout -> {holdout_ps[region]}")

    # --- pooled train ---
    df_pooled_loss = pd.concat(train_dfs_loss, ignore_index=True)
    df_pooled_full = pd.concat(train_dfs_full, ignore_index=True)

    print(f"\nPooled head pad: {months_head_pad}")
    print(f"Pooled tail pad: {months_tail_pad}")

    mbm.utils.seed_all(seed)

    ds_pooled = build_combined_LSTM_dataset(
        df_loss=df_pooled_loss,
        df_full=df_pooled_full,
        monthly_cols=monthly_cols,
        static_cols=static_cols,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        normalize_target=True,
        expect_target=True,
        show_progress=True,
    )

    train_idx, val_idx = mbm.data_processing.MBSequenceDataset.split_indices(
        len(ds_pooled), val_ratio=val_ratio, seed=seed)

    # --- save ---
    joblib.dump(ds_pooled, train_p, compress=3)
    joblib.dump({
        "train_idx": train_idx,
        "val_idx": val_idx
    },
                split_p,
                compress=3)
    print(f"\nSaved pooled train  -> {train_p}")
    print(f"Saved split indices -> {split_p}")
    print(f"Pooled dataset: {len(ds_pooled)} sequences "
          f"({len(train_idx)} train | {len(val_idx)} val)")

    return (
        {
            "ds_train": ds_pooled,
            "train_idx": train_idx,
            "val_idx": val_idx
        },
        holdout_datasets,
    )

In [ ]:
# ================================================================
# Multi-region pooled transformer training
# ================================================================

# --- 1. Build pooled assets (all three regions, holdouts excluded) ---
pooled_assets, ds_holdout_by_region = build_pooled_assets_from_cache(
    res_xreg_by_source=res_xreg_by_source,
    splits=splits,
    source_regions=SOURCE_REGIONS,  # ['CH', 'NOR', 'ISL']
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    seed=cfg.seed,
    val_ratio=0.2,
    cache_dir=CACHE_DIR,
    force_recompute=False,
)


In [ ]:
ds_pooled = pooled_assets["ds_train"]
train_idx = pooled_assets["train_idx"]
val_idx = pooled_assets["val_idx"]

# ---- 1. Sizes ----
print("=== Sizes ===")
print(f"Pooled dataset total sequences : {len(ds_pooled)}")
print(f"Train split                    : {len(train_idx)}")
print(f"Val split                      : {len(val_idx)}")
print(
    f"Sum matches total              : {len(train_idx) + len(val_idx) == len(ds_pooled)}"
)

# ---- 2. Tensor shapes ----
print("\n=== Tensor shapes ===")
print(
    f"Xm (monthly inputs) : {tuple(ds_pooled.Xm.shape)}  expected (N, T, {len(MONTHLY_COLS)})"
)
print(
    f"Xs (static inputs)  : {tuple(ds_pooled.Xs.shape)}  expected (N, {len(STATIC_COLS)})"
)
print(f"mv (valid mask)     : {tuple(ds_pooled.mv.shape)}")
print(f"y  (targets)        : {tuple(ds_pooled.y.shape)}")

# ---- 3. Pristine check (no scalers fitted yet) ----
print("\n=== Scaler state (should all be None) ===")
print(f"month_mean  : {ds_pooled.month_mean}")
print(f"static_mean : {ds_pooled.static_mean}")
print(f"y_mean      : {ds_pooled.y_mean}")

# ---- 4. NaN / Inf check ----
print("\n=== NaN / Inf check ===")
for name, t in [("Xm", ds_pooled.Xm), ("Xs", ds_pooled.Xs),
                ("y", ds_pooled.y)]:
    n_nan = torch.isnan(t).sum().item()
    n_inf = torch.isinf(t).sum().item()
    status = "OK" if (n_nan == 0 and n_inf == 0) else "ISSUE"
    print(f"  {name}: NaNs={n_nan}, Infs={n_inf}  [{status}]")

# ---- 5. Region coverage — check all three regions appear in keys ----
print("\n=== Region coverage in keys ===")
glaciers_in_pooled = {k[0] for k in ds_pooled.keys}
for region in SOURCE_REGIONS:
    df_loss = res_xreg_by_source[region]['data_monthly']
    train_glaciers = set(df_loss['GLACIER'].unique()) - set(
        splits[region]['holdout_glaciers'])
    overlap = glaciers_in_pooled & train_glaciers
    print(
        f"  {region}: {len(overlap)} / {len(train_glaciers)} train glaciers present in pooled dataset"
    )

# ---- 6. Holdout check — holdout glaciers must NOT appear in pooled ----
print("\n=== Holdout isolation check ===")
for region in SOURCE_REGIONS:
    holdout_gls = set(splits[region]['holdout_glaciers'])
    leaked = glaciers_in_pooled & holdout_gls
    status = "OK" if len(leaked) == 0 else f"LEAK: {leaked}"
    print(f"  {region}: {status}")

# ---- 7. Holdout datasets ----
print("\n=== Holdout datasets ===")
for region, ds_h in ds_holdout_by_region.items():
    n_w = ds_h.iw.sum().item()
    n_a = ds_h.ia.sum().item()
    holdout_gls = {k[0] for k in ds_h.keys}
    print(f"  {region}: {len(ds_h)} sequences ({n_w} winter | {n_a} annual) | "
          f"{len(holdout_gls)} glaciers")

# ---- 8. Winter / annual balance in pooled ----
print("\n=== Winter / Annual balance in pooled dataset ===")
n_w = ds_pooled.iw.sum().item()
n_a = ds_pooled.ia.sum().item()
print(f"  Winter : {n_w} ({100*n_w/len(ds_pooled):.1f}%)")
print(f"  Annual : {n_a} ({100*n_a/len(ds_pooled):.1f}%)")

## Grid search:

In [ ]:
# ================================================================
# Random grid search — Transformer on pooled CH+NOR+ISL
# ================================================================

import csv
import hashlib
import json
import random
from itertools import product


def generate_transformer_param_sets(const_params, param_grid):
    """
    Generate all valid transformer parameter combinations.
    Filters out combinations where nhead does not divide d_model.
    """
    keys = list(param_grid.keys())
    values = [param_grid[k] for k in keys]

    param_sets = []
    for combo in product(*values):
        params = dict(zip(keys, combo))

        # nhead must divide d_model
        if params["d_model"] % params["nhead"] != 0:
            continue

        # dim_feedforward should be >= d_model (not strictly required but sensible)
        if params["dim_feedforward"] < params["d_model"]:
            continue

        # no static MLP -> ignore static_hidden & static_dropout
        if params["static_layers"] == 0:
            params["static_hidden"] = None
            params["static_dropout"] = None

        full_params = {**const_params, **params}
        param_sets.append(full_params)

    return param_sets


def sample_transformer_param_sets(const_params,
                                  param_grid,
                                  n_samples,
                                  seed=42):
    all_params = generate_transformer_param_sets(const_params, param_grid)
    print(f"Total valid combinations: {len(all_params)}")
    if n_samples >= len(all_params):
        print(f"Requested {n_samples}, using all {len(all_params)}.")
        return all_params
    rng = random.Random(seed)
    return rng.sample(all_params, n_samples)


# --- pooled dataset (already built, excluding holdouts) ---
# reuse pooled_assets from earlier
ds_pooled = pooled_assets["ds_train"]
train_idx = pooled_assets["train_idx"]
val_idx = pooled_assets["val_idx"]

const_params = {
    "Fm": ds_pooled.Xm.shape[-1],  # 8
    "Fs": ds_pooled.Xs.shape[-1],  # 3
    "two_heads": False,
    "loss_spec": None,
    "T_max": 32,
}

param_grid = {
    # --- transformer encoder ---
    "d_model": [32, 64, 128, 256],
    "nhead": [2, 4, 8],  # filtered: must divide d_model
    "num_layers": [1, 2, 3, 4],
    "dim_feedforward": [64, 128, 256],  # filtered: must be >= d_model
    "dropout": [0.0, 0.1, 0.2],

    # --- static MLP ---
    "static_layers": [1, 2],
    "static_hidden": [64, 128],
    "static_dropout": [0.0, 0.1, 0.2],

    # --- head ---
    "head_dropout": [0.0, 0.05, 0.1],

    # --- optimisation ---
    "lr": [5e-4, 1e-3, 2e-3],
    "weight_decay": [1e-5, 1e-4],
}

N_SAMPLES = 100
sampled_params = sample_transformer_param_sets(
    const_params,
    param_grid,
    n_samples=N_SAMPLES,
    seed=cfg.seed,
)
print(f"Running {len(sampled_params)} random configurations")

In [ ]:
# --- grid search loop ---
RUN = True
if RUN:
    gs_models_dir = BASE_DIR / "models/Transformer_GS"
    gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
    os.makedirs(gs_models_dir, exist_ok=True)
    os.makedirs(gs_logs_dir, exist_ok=True)

    log_filename = gs_logs_dir / f"transformer_gs_pooled_{datetime.now().strftime('%Y-%m-%d')}.csv"

    fieldnames = list(
        sampled_params[0].keys()) + ["valid_loss", "val_rmse_a", "val_rmse_w"]
    with open(log_filename, mode="w", newline="") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()

    best_overall = {"val": float("inf"), "row": None, "params": None}

    for i, params in enumerate(sampled_params):
        mbm.utils.seed_all(cfg.seed)

        run_id = hashlib.md5(json.dumps(
            params, sort_keys=True).encode()).hexdigest()[:8]
        model_filename = str(gs_models_dir /
                             f"transformer_pooled_gs_{run_id}.pt")

        if os.path.exists(model_filename):
            os.remove(model_filename)

        print(f"\n--- Config {i+1}/{len(sampled_params)}  [id={run_id}] ---")
        print(params)

        # --- loaders: fit scalers on train split, evaluate on val split ---
        ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ds_pooled)
        # val dataset for metric evaluation (separate clone)
        ds_val_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            ds_pooled)

        train_dl, val_dl = ds_train_copy.make_loaders(
            train_idx=train_idx,
            val_idx=val_idx,
            batch_size_train=64,
            batch_size_val=128,
            seed=cfg.seed,
            fit_and_transform=True,
            shuffle_train=True,
            use_weighted_sampler=True,
            verbose=False,
        )

        # val loader with train scalers (for RMSE evaluation)
        val_eval_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_val_copy, ds_train_copy, batch_size=128, seed=cfg.seed)

        # --- build model & train ---
        model = mbm.models.Transformer_MB.build_model_from_params(
            cfg, params, device, verbose=False)
        loss_fn = mbm.models.Transformer_MB.resolve_loss_fn(params)

        history, best_val, _ = model.train_loop(
            device=device,
            train_dl=train_dl,
            val_dl=val_dl,
            epochs=150,
            lr=params["lr"],
            weight_decay=params["weight_decay"],
            clip_val=1,
            sched_factor=0.5,
            sched_patience=6,
            sched_threshold=0.01,
            sched_threshold_mode="rel",
            sched_cooldown=1,
            sched_min_lr=1e-6,
            es_patience=15,
            es_min_delta=1e-4,
            log_every=10,
            verbose=False,
            save_best_path=model_filename,
            loss_fn=loss_fn,
        )

        # --- evaluate on val split for RMSE (not used for selection, just logged) ---
        best_state = torch.load(model_filename, map_location=device)
        model.load_state_dict(best_state)

        val_metrics, _ = model.evaluate_with_preds(device, val_eval_dl,
                                                   ds_val_copy)
        val_rmse_a = val_metrics["RMSE_annual"]
        val_rmse_w = val_metrics["RMSE_winter"]

        print(
            f"  val_loss={best_val:.4f}  RMSE_a={val_rmse_a:.3f}  RMSE_w={val_rmse_w:.3f}"
        )

        # --- log ---
        row = {
            **params,
            "valid_loss": float(best_val),
            "val_rmse_a": float(val_rmse_a),
            "val_rmse_w": float(val_rmse_w),
        }
        with open(log_filename, mode="a", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)

        if best_val < best_overall["val"]:
            best_overall = {"val": best_val, "row": row, "params": params}

        # clean up model file to save disk space
        # comment out if you want to keep all checkpoints
        if os.path.exists(model_filename):
            os.remove(model_filename)

    print("\n=== Best config by validation loss ===")
    print(best_overall["params"])
    print(f"  val_loss={best_overall['val']:.4f}")
    print(f"  val_rmse_a={best_overall['row']['val_rmse_a']:.3f}")
    print(f"  val_rmse_w={best_overall['row']['val_rmse_w']:.3f}")

In [ ]:
gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))

# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[0]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

In [ ]:
K = 10

top_k = logs_gs_transformer.sort_values("valid_loss").head(K)

# --- top K table ---
print(f"=== Top {K} configs by validation loss ===")
print(top_k[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]].to_string(index=False))

# --- variability summary ---
param_cols = [
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay"
]

print(f"\n=== Parameter variability across top {K} ===")
for col in param_cols:
    vals = top_k[col].dropna()
    unique = sorted(vals.unique())
    if len(unique) == 1:
        stability = "stable"
    elif len(unique) <= 3:
        stability = "low variance"
    else:
        stability = "high variance"
    print(f"  {col:20s}: {unique}  [{stability}]")

# --- value counts for categorical-like params ---
print(f"\n=== Value counts in top {K} ===")
for col in [
        "d_model", "nhead", "num_layers", "dim_feedforward", "static_layers",
        "static_hidden"
]:
    counts = top_k[col].value_counts().sort_index()
    print(f"  {col:20s}: {counts.to_dict()}")

# --- loss spread ---
print(f"\n=== Loss spread across top {K} ===")
print(f"  valid_loss : min={top_k['valid_loss'].min():.4f}  "
      f"max={top_k['valid_loss'].max():.4f}  "
      f"range={top_k['valid_loss'].max() - top_k['valid_loss'].min():.4f}")
print(f"  val_rmse_a : min={top_k['val_rmse_a'].min():.3f}  "
      f"max={top_k['val_rmse_a'].max():.3f}")
print(f"  val_rmse_w : min={top_k['val_rmse_w'].min():.3f}  "
      f"max={top_k['val_rmse_w'].max():.3f}")